# Lab09 — Clean up: leave nothing running

**Storyline.** The workshop project holds an Agent Runtime instance, two Cloud Run services, an Agent Gateway,
Model Armor templates, a BigQuery dataset, registry entries and (if you created one) an online monitor. Everything
was created inside one project on purpose: shutting the project down removes all of it in one step.

**You will learn**
1. What shutting down a project does, and why billing is unlinked first
2. How to keep the project and remove only the pieces that run

Estimated time: 5 minutes.
Measured run time (all cells, fresh project, September 2026): 1 min; reading and exploring adds to it.

> **Terminal or notebook — your choice.** Every cell that calls a CLI prints the exact command first (`$ …`).
> Copy it into your own terminal (from the repo root) if you prefer to run it yourself; the cells just automate the same commands.


In [ ]:
# --- Workshop configuration (same cell at the top of every lab) ---
import os, sys, json, pathlib
from dotenv import load_dotenv

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
ENV_FILE = REPO_ROOT / "workshop.env"
assert ENV_FILE.exists(), "workshop.env not found - run Lab00 first"
load_dotenv(ENV_FILE, override=True)

PROJECT_ID      = os.environ["PROJECT_ID"]
PROJECT_NUMBER  = os.environ["PROJECT_NUMBER"]
REGION          = os.environ["REGION"]           # europe-west1: Agent Runtime, Sessions, Memory Bank, Gateway, Model Armor
MODEL_LOCATION  = os.environ["MODEL_LOCATION"]   # eu: multi-region endpoint that serves gemini-3.8-flash
MODEL           = os.environ["MODEL"]            # gemini-3.8-flash
AGENT_NAME      = os.environ["AGENT_NAME"]       # nova-assistant
AGENT_DIR       = REPO_ROOT / AGENT_NAME

# Make every shell (!) and SDK call in this notebook target the workshop project.
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": MODEL_LOCATION,
    "GOOGLE_GENAI_USE_VERTEXAI": "true",
    "CLOUDSDK_CORE_PROJECT": PROJECT_ID,
    "CLOUDSDK_CORE_DISABLE_PROMPTS": "1",
})
print(f"Project: {PROJECT_ID} ({PROJECT_NUMBER}) | region: {REGION} | model: {MODEL} @ {MODEL_LOCATION}")
print(f"Agent dir: {AGENT_DIR}")

def terminal(cmd, cwd=None):
    """Print the exact command a cell is about to run, so you can copy/paste it into your own terminal."""
    prefix = f"cd {os.path.relpath(cwd, REPO_ROOT)} && " if cwd else ""
    print(f"$ {prefix}{cmd}\n")

## 9.1 What shutting down a project does

The Resource Manager documentation states it directly: *"Shutting down a project stops all billing and resource usage."*
The timeline:

| When | What happens |
| --- | --- |
| immediately | the project is marked for deletion and becomes unusable; *"Any associated Cloud Billing account is disconnected"* |
| 30 days | recovery window (`lifecycleState: DELETE_REQUESTED`); an Owner can restore it with `gcloud projects undelete` |
| after 30 days | *"the project and all its resources are permanently deleted and can't be recovered"* |

Two details worth knowing before you press the button:

1. **Unlink billing first.** The same page notes that *"Projects may continue to incur charges until the current billing
   cycle ends"* and recommends disabling billing before the shutdown. The cell below does exactly that: unlink, then delete.
2. **The project id is gone for good.** Ids are never reused, and the project counts against your project quota until the
   30 days are over. Lab00 puts the date into the id so the next run gets a fresh one.

Nothing the workshop created lives outside the project: the organization-policy override from Lab06, the IAM access
policies and the registry entries are all project-scoped. What stays on your machine costs nothing: the `geap-workshop`
gcloud configuration, `workshop.env`, the `nova-assistant/` folder and the coding-agent skills from Lab00.

Docs: [Shut down (delete) and restore projects](https://docs.cloud.google.com/resource-manager/docs/delete-restore-projects).


## 9.2 Option A — shut down the project

If you want to try the optional [Lab08](lab08_gemini_enterprise.ipynb) (Gemini Enterprise), do it first.
The cell is a dry run until you set `CONFIRM_DELETE = True`.


In [ ]:
# --- Shut down the project: unlink billing -> delete the project -> confirm the state -> switch gcloud back ---
import subprocess

# Safety catch: nothing below runs until you set this to True.
CONFIRM_DELETE = False

steps = [
    f"gcloud billing projects unlink {PROJECT_ID}",                              # stop charges now, not at the end of the billing cycle
    f"gcloud projects delete {PROJECT_ID} --quiet",                              # marks the project for deletion (30-day recovery window)
    f"gcloud projects describe {PROJECT_ID} --format='value(lifecycleState)'",  # expect DELETE_REQUESTED
    "gcloud config configurations activate default",                            # back to your usual gcloud setup
]
for cmd in steps:
    terminal(cmd)
    if CONFIRM_DELETE:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print((r.stdout or r.stderr).strip(), "\n")

if not CONFIRM_DELETE:
    print("dry run - set CONFIRM_DELETE = True and re-run this cell to execute the four commands")
else:
    print(f"project {PROJECT_ID} is shutting down; restore within 30 days with: gcloud projects undelete {PROJECT_ID}")


## 9.3 Option B — keep the project, remove what runs

Delete the pieces that hold compute or storage; configuration (registry entries, IAM access policies, Model Armor
templates, org-policy override) has no running cost and can stay. From the repo root:

```bash
source workshop.env
TOKEN=$(gcloud auth print-access-token)

# Agent Runtime instance, with its sessions, memories and sandboxes (Lab02-06). force=true deletes the child resources.
curl -s -X DELETE "${NOVA_AGENT_URL}?force=true" -H "Authorization: Bearer ${TOKEN}"

# Online monitor, if you created one in Lab07 §7.7 (list first, then delete by id).
API="https://${REGION}-aiplatform.googleapis.com/v1beta1/projects/${PROJECT_ID}/locations/${REGION}"
curl -s "${API}/onlineEvaluators" -H "Authorization: Bearer ${TOKEN}"
curl -s -X DELETE "${API}/onlineEvaluators/MONITOR_ID" -H "Authorization: Bearer ${TOKEN}"

# The two mock enterprise systems on Cloud Run (Lab03, Lab06).
gcloud run services delete nova-inventory-mcp --region=${REGION} --project=${PROJECT_ID} --quiet
gcloud run services delete nova-returns-agent --region=${REGION} --project=${PROJECT_ID} --quiet

# Agent Gateway and its policies and extensions (Lab06).
gcloud network-security authz-policies delete nova-iap-authz-policy --location=${REGION} --project=${PROJECT_ID} --quiet
gcloud network-security authz-policies delete nova-ma-content-policy --location=${REGION} --project=${PROJECT_ID} --quiet
gcloud beta service-extensions authz-extensions delete nova-iap-authz-ext --location=${REGION} --project=${PROJECT_ID} --quiet
gcloud beta service-extensions authz-extensions delete nova-ma-content-ext --location=${REGION} --project=${PROJECT_ID} --quiet
gcloud network-services agent-gateways delete nova-egress-gateway --location=${REGION} --project=${PROJECT_ID} --quiet

# Sample data in BigQuery (Lab03).
bq rm -r -f -d ${PROJECT_ID}:nova_shop

# Private skill in Agent Registry (Lab03): revisions first, then the skill.
gcloud alpha agent-registry skills revisions list --skill=private-nova-sales-analytics --location=${NOVA_SKILLS_LOCATION} --project=${PROJECT_ID}
gcloud alpha agent-registry skills revisions delete REVISION_ID --skill=private-nova-sales-analytics --location=${NOVA_SKILLS_LOCATION} --project=${PROJECT_ID} --quiet
gcloud alpha agent-registry skills delete private-nova-sales-analytics --location=${NOVA_SKILLS_LOCATION} --project=${PROJECT_ID} --quiet
```

Coding-agent prompt: *"Delete the Nova Assistant runtime instance, the two Cloud Run mocks and the egress gateway in the
workshop project, and list what is still there."*


## Recap — the whole journey

| Lab | Nova Assistant gained | Platform capability |
| --- | --- | --- |
| 01 | tools, instruction, local testing | ADK, Agents CLI, playground |
| 02 | a home in the cloud, persistent sessions, discoverability | Agent Runtime, Agent Registry |
| 03 | real data, a warehouse connection, domain skills, an analyst | BigQuery remote MCP server, custom MCP server on Cloud Run, Agent Registry, Skills |
| 04 | memory, safe computation, feedback, transparency | Sessions, Memory Bank, Code Execution, Feedback service, metrics, logs, traces |
| 05 | protection against injection and leaks | Model Armor (two templates, plugin, floor settings), alerts |
| 06 | governed access to enterprise systems, Model Armor on the network path | Agent Identity, Agent Gateway, IAM access policies, Model Armor `CONTENT_AUTHZ` |
| 07 | proof of quality | evaluation datasets, judges, user simulation, compare |
| 08 *(optional)* | a place in the employee-facing app | Gemini Enterprise (ADK and A2A registration) |
| 09 | a clean exit | project shutdown, billing unlink, resource clean-up |
